In [1]:
import glob
import numpy
import pandas
import seaborn
import matplotlib.pyplot as plt
import multiprocessing as mp
import os
from build import build_model
import coralme
from tqdm import tqdm

In [2]:
from IPython.display import display, HTML, Math, Markdown
display(HTML("<style>.container { width:95% !important; }</style>"))

%load_ext autoreload
%autoreload 2

### Load

In [3]:
def load_data(path):
    try:
        df = pandas.read_csv(path,index_col=0)[["fluxes"]]
    except:
        df = pandas.DataFrame(columns=["fluxes"])
    df.columns = [org]
    return df

In [4]:
dataset = ""

In [5]:
conditions = ["base"]
organisms = set(pandas.read_csv("biomass_constrained.txt",index_col=0,header=None).index.to_list())
fluxes = {i:pandas.DataFrame() for i in conditions}
for org in tqdm(organisms):
    for c in conditions:
        tmp = load_data("./cases/fluxes/{}{}/{}.csv".format(c,dataset,org))
        tmp = tmp.loc[[i for i in tmp.index if i.startswith("EX_")]]
        fluxes[c] = pandas.concat([fluxes[c],tmp], axis=1).fillna(0)

  0%|          | 0/495 [00:00<?, ?it/s]

100%|██████████| 495/495 [00:02<00:00, 195.30it/s]


In [8]:
df = fluxes["base"]

In [14]:
df[df.index.str.contains("phe")]

,Burkholderiales_bacterium_1_1_47,Alistipes_indistinctus_YIT_12060,Clostridium_botulinum_A_str_ATCC_19397,Lactobacillus_casei_casei_BL23,Kytococcus_sedentarius_DSM_20547,Staphylococcus_haemolyticus_JCSC1435,Marvinbryantia_formatexigens_I_52_DSM_14469,Fusobacterium_nucleatum_subsp_vincentii_3_1_36A2,Actinomyces_graevenitzii_C83,Alistipes_onderdonkii_DSM_19147,...,Capnocytophaga_sputigena_ATCC_33612,Fusobacterium_necrophorum_D12,Enterococcus_hirae_ATCC_9790,Clostridium_scindens_ATCC_35704,Mitsuokella_multacida_DSM_20544,Bacteroides_sp_20_3,Actinomyces_odontolyticus_ATCC_17982,Fusobacterium_periodonticum_1_1_41FAA,Bacillus_halodurans_C_125,Lactobacillus_fermentum_ATCC_14931
EX_glyphe(e),0.000000,0.000000,0.0,0.000000,0.000000,2.247401e-23,0.000000,0.000000,0.000000,0.000000,...,0.000000,3.207486e-23,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
EX_phe_L(e),-0.001534,0.929916,-1.0,-0.039523,-0.009068,-7.794717e-02,-0.020282,-0.028920,-0.011264,-0.031487,...,-0.032291,-5.907676e-02,-0.065012,-0.019832,0.776779,0.0,0.976674,-0.022706,-0.093347,-0.024659
EX_pheme(e),0.000000,-0.000449,0.0,0.000000,-0.000373,-3.074259e-03,0.000000,-0.000234,-0.000322,-0.000422,...,-0.000450,0.000000e+00,-0.002473,0.000000,0.000000,0.0,0.000000,0.000000,-0.004204,0.000000
EX_34dhphe(e),0.000000,0.000000,0.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
EX_phenol(e),0.000000,0.000000,0.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000


In [17]:
model = coralme.io.pickle.load_pickle_me_model("me-models/Alistipes_indistinctus_YIT_12060/MEModel-step3-Alistipes_indistinctus_YIT_12060-ME-TS.pkl")

Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-03
Read LP format model from file /tmp/tmpowjbcp8g.lp
Reading time = 0.00 seconds
: 0 rows, 0 columns, 0 nonzeros
Read LP format model from file /tmp/tmpwrfpsta6.lp
Reading time = 0.01 seconds
: 1091 rows, 2444 columns, 10636 nonzeros
Read LP format model from file /tmp/tmpno2qpcu6.lp
Reading time = 0.00 seconds
: 1104 rows, 2442 columns, 10466 nonzeros


In [20]:
flux_dict = pandas.read_csv("cases/fluxes/base/Alistipes_indistinctus_YIT_12060.csv",index_col=0)["fluxes"].to_dict()

In [23]:
from coralme.builder.helper_functions import flux_based_reactions

In [30]:
flux_based_reactions(model, "phe__L_c", flux_dict=flux_dict, only_types=["MetabolicReaction"])

,lb,ub,rxn_flux,met_flux,reaction
PHETA1_REV_742725.3.peg.418-MONOMER,0.0,1000.0,0.959113,0.959113,[4.17305788963832e-6*mu] 742725.3.peg.418-MONO...
PHEt2r_REV_CPLX_dummy,0.0,1000.0,0.929916,-0.929916,[4.27350427350427e-6*mu] CPLX_dummy + 1.0 h_c ...
GLYPHEHYc_REV_g.27472.CDS.1139-MONOMER,0.0,1000.0,0.000000,0.000000,[2.01597441815508e-6*mu] g.27472.CDS.1139-MONO...
GLYPHEHYc_FWD_g.27472.CDS.1139-MONOMER,0.0,1000.0,0.000000,0.000000,[2.01597441815508e-6*mu] g.27472.CDS.1139-MONO...
GLYPHEHYc_FWD_g.27472.CDS.332-MONOMER_mod_fe2(2),0.0,1000.0,0.000000,0.000000,[5.20754514983025e-6*mu] g.27472.CDS.332-MONOM...
GLYPHEHYc_REV_g.27472.CDS.332-MONOMER_mod_fe2(2),0.0,1000.0,0.000000,0.000000,[5.20754514983025e-6*mu] g.27472.CDS.332-MONOM...
GLYPHEHYc_REV_g.27472.CDS.2400-MONOMER,0.0,1000.0,0.000000,0.000000,[3.27306021477841e-6*mu] g.27472.CDS.2400-MONO...
GLYPHEHYc_FWD_g.27472.CDS.2400-MONOMER,0.0,1000.0,0.000000,0.000000,[3.27306021477841e-6*mu] g.27472.CDS.2400-MONO...
GLYPHEHYc_FWD_g.27472.CDS.1586-MONOMER,0.0,1000.0,0.000000,0.000000,[1.96806830296169e-6*mu] g.27472.CDS.1586-MONO...
GLYPHEHYc_REV_g.27472.CDS.1586-MONOMER,0.0,1000.0,0.000000,0.000000,[1.96806830296169e-6*mu] g.27472.CDS.1586-MONO...


In [26]:
flux_based_reactions(model, "glu__L_c", flux_dict=flux_dict)

,lb,ub,rxn_flux,met_flux,reaction
GLUDy_REV_742725.3.peg.202-MONOMER,0.0,1000.0,7.485648,7.485648,[3.48705587255663e-6*mu] 742725.3.peg.202-MONO...
ASPTA_REV_742725.3.peg.922-MONOMER,0.0,1000.0,5.879233,-5.879233,[3.64075605142997e-6*mu] 742725.3.peg.922-MONO...
PHETA1_REV_742725.3.peg.418-MONOMER,0.0,1000.0,0.959113,-0.959113,[4.17305788963832e-6*mu] 742725.3.peg.418-MONO...
VALTA_REV_742725.3.peg.715-MONOMER,0.0,1000.0,0.615429,-0.615429,1.0 3mob_c + [4.19105628069294e-6*mu] 742725.3...
GLUDC_FWD_g.27472.CDS.1985-MONOMER_mod_pydx5p(1),0.0,1000.0,0.236203,-0.236203,[7.94315688153074e-7*mu] g.27472.CDS.1985-MONO...
...,...,...,...,...,...
ANS_FWD_742725.3.peg.1750-MONOMER,0.0,1000.0,0.000000,0.000000,[6.49766436076218e-6*mu] 742725.3.peg.1750-MON...
translation_g.27472.CDS.1090,0.0,1000.0,0.000000,0.000000,1.0 10fthf5glu_c + [2.55589556001079e-8*mu + 1...
translation_g.27472.CDS.1921,0.0,1000.0,0.000000,0.000000,1.0 10fthf5glu_c + [2.55589556001079e-8*mu + 1...
translation_g.27472.CDS.1718,0.0,1000.0,0.000000,0.000000,1.0 10fthf5glu_c + [2.55589556001079e-8*mu + 1...


In [27]:
flux_based_reactions(model, "phpyr_c", flux_dict=flux_dict)

,lb,ub,rxn_flux,met_flux,reaction
PHETA1_REV_742725.3.peg.418-MONOMER,0.0,1000.0,0.959113,-0.959113,[4.17305788963832e-6*mu] 742725.3.peg.418-MONO...
PPNDH_FWD_742725.3.peg.1355-MONOMER,0.0,1000.0,0.959113,0.959113,[2.75748887525285e-6*mu] 742725.3.peg.1355-MON...
PHETA1_FWD_742725.3.peg.418-MONOMER,0.0,1000.0,0.000000,0.000000,[4.17305788963832e-6*mu] 742725.3.peg.418-MONO...


In [34]:
model.get("chor_c").reactions

frozenset({<MetabolicReaction ADCS_FWD_742725.3.peg.1750-MONOMER at 0x77e397421900>,
           <MetabolicReaction ADCS_REV_742725.3.peg.1750-MONOMER at 0x77e3974217e0>,
           <MetabolicReaction ANS2_FWD_742725.3.peg.1750-MONOMER at 0x77e3974e1960>,
           <MetabolicReaction ANS2_FWD_742725.3.peg.1751-MONOMER at 0x77e3974e1a80>,
           <MetabolicReaction ANS2_REV_742725.3.peg.1750-MONOMER at 0x77e3974e17b0>,
           <MetabolicReaction ANS2_REV_742725.3.peg.1751-MONOMER at 0x77e3974e19f0>,
           <MetabolicReaction ANS_FWD_742725.3.peg.1750-MONOMER at 0x77e3974e15a0>,
           <MetabolicReaction ANS_FWD_742725.3.peg.1751-MONOMER at 0x77e3974e1750>,
           <MetabolicReaction ANS_REV_742725.3.peg.1750-MONOMER at 0x77e3974e14e0>,
           <MetabolicReaction ANS_REV_742725.3.peg.1751-MONOMER at 0x77e3974e1660>,
           <MetabolicReaction CHDHR_FWD_CPLX_dummy at 0x77e39735e260>,
           <MetabolicReaction CHDHR_REV_CPLX_dummy at 0x77e39735e1a0>,
           <

In [28]:
flux_based_reactions(model, "pphn_c", flux_dict=flux_dict)

,lb,ub,rxn_flux,met_flux,reaction
CHORM_FWD_742725.3.peg.1358-MONOMER,0.0,1000.0,0.998654,0.998654,[4.08314644260648e-6*mu] 742725.3.peg.1358-MON...
PPNDH_FWD_742725.3.peg.1355-MONOMER,0.0,1000.0,0.959113,-0.959113,[2.75748887525285e-6*mu] 742725.3.peg.1355-MON...
PPND2_FWD_g.27472.CDS.1445-MONOMER,0.0,1000.0,0.039541,-0.039541,[4.63645373408818e-6*mu] g.27472.CDS.1445-MONO...
LARGNAT_REV_742725.3.peg.1356-MONOMER,0.0,1000.0,0.000000,0.000000,[3.78173093492907e-6*mu] 742725.3.peg.1356-MON...
LARGNAT_REV_742725.3.peg.844-MONOMER,0.0,1000.0,0.000000,0.000000,[3.68346697669504e-6*mu] 742725.3.peg.844-MONO...
LARGNAT_FWD_742725.3.peg.1356-MONOMER,0.0,1000.0,0.000000,0.000000,[3.78173093492907e-6*mu] 742725.3.peg.1356-MON...
LARGNAT_REV_742725.3.peg.418-MONOMER,0.0,1000.0,0.000000,0.000000,[4.17305788963832e-6*mu] 742725.3.peg.418-MONO...
LARGNAT_REV_742725.3.peg.1641-MONOMER,0.0,1000.0,0.000000,0.000000,[3.48507578362304e-6*mu] 742725.3.peg.1641-MON...
LARGNAT_FWD_742725.3.peg.418-MONOMER,0.0,1000.0,0.000000,0.000000,[4.17305788963832e-6*mu] 742725.3.peg.418-MONO...
PPND_FWD_742725.3.peg.1357-MONOMER,0.0,1000.0,0.000000,0.000000,[4.63645373408818e-6*mu] 742725.3.peg.1357-MON...


In [29]:
flux_based_reactions(model, "chor_c", flux_dict=flux_dict)

,lb,ub,rxn_flux,met_flux,reaction
CHORt_FWD_CPLX_dummy,0.0,1000.0,1.000000,1.000000,[4.27350427350427e-6*mu] CPLX_dummy + 1.0 chor...
CHORM_FWD_742725.3.peg.1358-MONOMER,0.0,1000.0,0.998654,-0.998654,[4.08314644260648e-6*mu] 742725.3.peg.1358-MON...
ADCS_FWD_742725.3.peg.1750-MONOMER,0.0,1000.0,0.001346,-0.001346,[6.49766436076218e-6*mu] 742725.3.peg.1750-MON...
CHORS_FWD_742725.3.peg.1685-MONOMER_mod_fadh2(1),0.0,1000.0,0.000000,0.000000,1.0 3psme_c + [1.55781722558685e-6*mu] 742725....
ANS2_FWD_742725.3.peg.1750-MONOMER,0.0,1000.0,0.000000,0.000000,[6.49766436076218e-6*mu] 742725.3.peg.1750-MON...
ANS_FWD_742725.3.peg.1750-MONOMER,0.0,1000.0,0.000000,0.000000,[6.49766436076218e-6*mu] 742725.3.peg.1750-MON...
CHDHR_FWD_CPLX_dummy,0.0,1000.0,0.000000,0.000000,[4.27350427350427e-6*mu] CPLX_dummy + 1.0 chor...
ANS2_FWD_742725.3.peg.1751-MONOMER,0.0,1000.0,0.000000,0.000000,[3.2420251899272e-6*mu] 742725.3.peg.1751-MONO...
CHORS_FWD_742725.3.peg.1685-MONOMER_mod_fmnh2(1),0.0,1000.0,0.000000,0.000000,1.0 3psme_c + [1.56051258840809e-6*mu] 742725....
ANS_FWD_742725.3.peg.1751-MONOMER,0.0,1000.0,0.000000,0.000000,[3.2420251899272e-6*mu] 742725.3.peg.1751-MONO...


In [7]:
for i,df in fluxes.items():
    df.to_csv("datasets/1.ExchangeFluxes_{}.csv".format(i))